드라이브 마운트, 폴더 생성

In [2]:
from google.colab import drive
from pathlib import Path
import os, shutil, yaml, torch

drive.mount("/content/drive")

PROJECT_PATH = Path("/content/drive/MyDrive/Single_Flower")

DATASETS_DIR = PROJECT_PATH / "datasets"
RUNS_DIR = PROJECT_PATH / "training_results"
MODELS_DIR = PROJECT_PATH / "models"
VAL_DIR = PROJECT_PATH / "validation_results"
PREDICT_DIR = PROJECT_PATH / "prediction_results"
TEST_IMAGE_DIR = PROJECT_PATH / "test_images_single_flower"

for p in [PROJECT_PATH, DATASETS_DIR, RUNS_DIR, MODELS_DIR, VAL_DIR, PREDICT_DIR, TEST_IMAGE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("프로젝트 경로:", PROJECT_PATH)
print("테스트 이미지 폴더:", TEST_IMAGE_DIR)

Mounted at /content/drive
프로젝트 경로: /content/drive/MyDrive/Single_Flower
테스트 이미지 폴더: /content/drive/MyDrive/Single_Flower/test_images_single_flower


패키지 설치, GPU 확인

In [3]:
!pip install -q ultralytics roboflow pyyaml

import torch
from ultralytics import YOLO

print("CUDA 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU가 안 잡혔습니다. Colab 런타임을 GPU로 바꾸세요.")

!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 138.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
CUDA 사용 가능: True
GPU: Tesla T4
Wed May 27 10:18:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07   

Roboflow 다운로드

In [3]:
from google.colab import userdata
from roboflow import Roboflow
from pathlib import Path
import shutil

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

if not ROBOFLOW_API_KEY:
    raise ValueError("Colab Secrets에 ROBOFLOW_API_KEY를 등록하세요.")

WORKSPACE = "minseo-kim-nw4w2"
PROJECT = "flower-nneat"
VERSION = 1

DATASET_DRIVE_DIR = DATASETS_DIR / f"{PROJECT}-v{VERSION}-single-flower-yolo11"

FORCE_REDOWNLOAD = False

if FORCE_REDOWNLOAD and DATASET_DRIVE_DIR.exists():
    shutil.rmtree(DATASET_DRIVE_DIR)

if not (DATASET_DRIVE_DIR / "data.yaml").exists():
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    version = project.version(VERSION)

    dataset = version.download(
        "yolov11",
        location=str(DATASET_DRIVE_DIR)
    )

    print("다운로드 완료:", dataset.location)
else:
    print("이미 다운로드된 데이터셋 사용:", DATASET_DRIVE_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/drive/MyDrive/Single_Flower/datasets/flower-nneat-v1-single-flower-yolo11 in yolov11:: 100%|██████████| 8241/8241 [01:39<00:00, 83.15it/s]

다운로드 완료: /content/drive/MyDrive/Single_Flower/datasets/flower-nneat-v1-single-flower-yolo11


드라이브 데이터셋 로컬로 복사

In [4]:
LOCAL_DATASET_DIR = Path(f"/content/{PROJECT}_v{VERSION}_single_flower")

if LOCAL_DATASET_DIR.exists():
    shutil.rmtree(LOCAL_DATASET_DIR)

shutil.copytree(DATASET_DRIVE_DIR, LOCAL_DATASET_DIR)

print("로컬 복사 완료:", LOCAL_DATASET_DIR)
print("파일 목록:", os.listdir(LOCAL_DATASET_DIR))

로컬 복사 완료: /content/flower-nneat_v1_single_flower
파일 목록: ['train', 'test', 'README.roboflow.txt', 'data.yaml', 'README.dataset.txt', 'valid']


data.yaml single-class 검증, 수정

In [5]:
original_yaml_path = LOCAL_DATASET_DIR / "data.yaml"
fixed_yaml_path = LOCAL_DATASET_DIR / "data_fixed.yaml"

with open(original_yaml_path, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

print("원본 data.yaml:")
print(yaml.safe_dump(data, allow_unicode=True, sort_keys=False))

fixed_data = {
    "path": str(LOCAL_DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["flower"],
}

with open(fixed_yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(fixed_data, f, allow_unicode=True, sort_keys=False)

print("수정된 data_fixed.yaml:")
print(yaml.safe_dump(fixed_data, allow_unicode=True, sort_keys=False))

원본 data.yaml:
train: ../train/images
val: ../valid/images
test: ../test/images
nc: 1
names:
- flower
roboflow:
  workspace: minseo-kim-nw4w2
  project: flower-nneat
  version: 1
  license: Public Domain
  url: https://universe.roboflow.com/minseo-kim-nw4w2/flower-nneat/dataset/1

수정된 data_fixed.yaml:
path: /content/flower-nneat_v1_single_flower
train: train/images
val: valid/images
test: test/images
nc: 1
names:
- flower



이미지/라벨 개수 확인

In [6]:
from pathlib import Path

def count_files(path, exts):
    path = Path(path)
    if not path.exists():
        return 0
    return len([p for p in path.iterdir() if p.suffix.lower() in exts])

img_exts = {".jpg", ".jpeg", ".png", ".webp"}
label_exts = {".txt"}

for split in ["train", "valid", "test"]:
    img_count = count_files(LOCAL_DATASET_DIR / split / "images", img_exts)
    label_count = count_files(LOCAL_DATASET_DIR / split / "labels", label_exts)

    print(f"{split}: images={img_count}, labels={label_count}")

print("YAML:", fixed_yaml_path)

train: images=3660, labels=3660
valid: images=229, labels=229
test: images=229, labels=229
YAML: /content/flower-nneat_v1_single_flower/data_fixed.yaml


polygon -> bbox 변환

In [8]:
from pathlib import Path
import shutil

DATASET_DIR = LOCAL_DATASET_DIR

BACKUP_DIR = DATASET_DIR / "_labels_backup_before_polygon_to_bbox"

if not BACKUP_DIR.exists():
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)

    for split in ["train", "valid", "test"]:
        src = DATASET_DIR / split / "labels"
        dst = BACKUP_DIR / split / "labels"

        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)

    print("라벨 백업 완료:", BACKUP_DIR)
else:
    print("이미 백업 존재:", BACKUP_DIR)


def clamp01(x):
    return max(0.0, min(1.0, x))


def polygon_to_bbox(parts):
    """
    YOLO segmentation format:
    class x1 y1 x2 y2 x3 y3 ...

    YOLO detection format:
    class x_center y_center width height
    """
    cls_id = parts[0]
    coords = list(map(float, parts[1:]))

    xs = coords[0::2]
    ys = coords[1::2]

    x_min = clamp01(min(xs))
    x_max = clamp01(max(xs))
    y_min = clamp01(min(ys))
    y_max = clamp01(max(ys))

    w = x_max - x_min
    h = y_max - y_min

    if w <= 0 or h <= 0:
        return None

    x_center = x_min + w / 2
    y_center = y_min + h / 2

    return f"{cls_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"


converted_lines = 0
kept_lines = 0
dropped_lines = 0
bad_lines = []

for split in ["train", "valid", "test"]:
    labels_dir = DATASET_DIR / split / "labels"

    if not labels_dir.exists():
        continue

    for label_path in labels_dir.glob("*.txt"):
        original_text = label_path.read_text().strip()

        if not original_text:
            continue

        new_lines = []

        for line_idx, line in enumerate(original_text.splitlines(), start=1):
            parts = line.split()

            # 이미 YOLO bbox 형식이면 그대로 유지
            if len(parts) == 5:
                cls_id = parts[0]

                if cls_id != "0":
                    parts[0] = "0"

                new_lines.append(" ".join(parts))
                kept_lines += 1
                continue

            # polygon/segmentation 형식이면 bbox로 변환
            # class + 짝수 개 좌표여야 함
            if len(parts) > 5 and (len(parts) - 1) % 2 == 0:
                parts[0] = "0"

                try:
                    bbox_line = polygon_to_bbox(parts)

                    if bbox_line is None:
                        dropped_lines += 1
                    else:
                        new_lines.append(bbox_line)
                        converted_lines += 1

                except Exception as e:
                    bad_lines.append((str(label_path), line_idx, line, str(e)))
                    dropped_lines += 1

            else:
                bad_lines.append((str(label_path), line_idx, line, "알 수 없는 라벨 형식"))
                dropped_lines += 1

        label_path.write_text("\n".join(new_lines) + ("\n" if new_lines else ""))

print("기존 bbox 유지:", kept_lines)
print("polygon -> bbox 변환:", converted_lines)
print("drop된 라벨:", dropped_lines)
print("여전히 이상한 라벨:", len(bad_lines))

for item in bad_lines[:20]:
    print(item)

라벨 백업 완료: /content/flower-nneat_v1_single_flower/_labels_backup_before_polygon_to_bbox
기존 bbox 유지: 36353
polygon -> bbox 변환: 48
drop된 라벨: 0
여전히 이상한 라벨: 0


모든 class id가 0인지 검사

In [9]:
bad_labels = []
empty_labels = 0
total_boxes = 0

for split in ["train", "valid", "test"]:
    labels_dir = LOCAL_DATASET_DIR / split / "labels"

    if not labels_dir.exists():
        continue

    for label_path in labels_dir.glob("*.txt"):
        text = label_path.read_text().strip()

        if not text:
            empty_labels += 1
            continue

        for line_idx, line in enumerate(text.splitlines(), start=1):
            parts = line.split()

            if len(parts) != 5:
                bad_labels.append((str(label_path), line_idx, line, "컬럼 수 이상"))
                continue

            cls_id = parts[0]
            total_boxes += 1

            if cls_id != "0":
                bad_labels.append((str(label_path), line_idx, line, "class id가 0이 아님"))

print("총 bbox 수:", total_boxes)
print("빈 label 파일 수:", empty_labels)
print("문제 label 수:", len(bad_labels))

for item in bad_labels[:20]:
    print(item)

if bad_labels:
    raise ValueError("class id 또는 label 형식 문제가 있습니다. 위 출력 확인하세요.")

총 bbox 수: 36401
빈 label 파일 수: 0
문제 label 수: 0


학습 시작

In [10]:
from ultralytics import YOLO
from pathlib import Path
import torch

assert torch.cuda.is_available(), "GPU가 안 잡혔습니다. Colab 런타임을 GPU로 바꾸세요."

DATA_YAML = str(fixed_yaml_path)

RUN_NAME = f"single_flower_yolo11s_v{VERSION}_704_baseline"

model = YOLO("yolo11s.pt")

results = model.train(
    data=DATA_YAML,

    epochs=80,
    patience=20,

    imgsz=704,
    batch=16,
    workers=4,
    device=0,

    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=False,

    pretrained=True,
    optimizer="auto",
    cos_lr=True,

    # color augmentation OFF
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,

    # geometry augmentation OFF
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,

    # flip OFF for baseline
    fliplr=0.0,
    flipud=0.0,

    # mosaic/mixup OFF
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    cache=False,
    amp=True,
    plots=True,
    save_period=10,

    seed=42,
    deterministic=True,
)

Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/flower-nneat_v1_single_flower/data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=704, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=single_flower_yolo11s_v1_704_baseline, nbs=64, nms=False, opset=None, optimize=Fals

학습 결과 위치 확인

In [11]:
RUN_DIR = RUNS_DIR / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

print("RUN_DIR:", RUN_DIR)
print("best.pt exists:", BEST_PT.exists(), BEST_PT)
print("last.pt exists:", LAST_PT.exists(), LAST_PT)

RUN_DIR: /content/drive/MyDrive/Single_Flower/training_results/single_flower_yolo11s_v1_704_baseline
best.pt exists: True /content/drive/MyDrive/Single_Flower/training_results/single_flower_yolo11s_v1_704_baseline/weights/best.pt
last.pt exists: True /content/drive/MyDrive/Single_Flower/training_results/single_flower_yolo11s_v1_704_baseline/weights/last.pt


중간에 끊겼을 때 이어서

In [ ]:
from ultralytics import YOLO
from pathlib import Path

LAST_PT = Path("/content/drive/MyDrive/Single_Flower/training_results/여기에-run-folder/weights/last.pt")

print("last.pt exists:", LAST_PT.exists())
print(LAST_PT)

model = YOLO(str(LAST_PT))
results = model.train(resume=True)

validation / test 평가

In [12]:
from ultralytics import YOLO

BEST_PT = RUN_DIR / "weights" / "best.pt"

model = YOLO(str(BEST_PT))

val_results = model.val(
    data=str(fixed_yaml_path),
    split="val",
    imgsz=704,
    batch=16,
    device=0,
    project=str(VAL_DIR),
    name=f"{RUN_NAME}_val",
    exist_ok=True,
)

test_results = model.val(
    data=str(fixed_yaml_path),
    split="test",
    imgsz=704,
    batch=16,
    device=0,
    project=str(VAL_DIR),
    name=f"{RUN_NAME}_test",
    exist_ok=True,
)

Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1564.9±306.2 MB/s, size: 62.5 KB)
val: Scanning /content/flower-nneat_v1_single_flower/valid/labels.cache... 229 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 229/229 43.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 15/15 2.2it/s 6.9s
                   all        229       2581      0.731      0.691      0.724      0.437
Speed: 3.0ms preprocess, 13.0ms inference, 0.0ms loss, 2.2ms postprocess per image
Results saved to /content/drive/MyDrive/Single_Flower/validation_results/single_flower_yolo11s_v1_704_baseline_val
Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 24.6±6.6 MB/s, size: 50.9 KB)
val: Scanning /con

최종 모델 복사

In [13]:
import shutil
from pathlib import Path

FINAL_MODEL = MODELS_DIR / f"{RUN_NAME}_best.pt"

shutil.copy2(BEST_PT, FINAL_MODEL)

print("최종 모델 복사 완료:")
print(FINAL_MODEL)

최종 모델 복사 완료:
/content/drive/MyDrive/Single_Flower/models/single_flower_yolo11s_v1_704_baseline_best.pt


테스트

In [14]:
from ultralytics import YOLO
from pathlib import Path
from IPython.display import Image, display

BEST_PT = Path("/content/drive/MyDrive/Single_Flower/training_results/single_flower_yolo11s_v1_704_baseline/weights/best.pt")

TEST_IMAGE_DIR = Path("/content/drive/MyDrive/Single_Flower/test_images_single_flower")
PREDICT_DIR = Path("/content/drive/MyDrive/Single_Flower/prediction_results")

model = YOLO(str(BEST_PT))

results = model.predict(
    source=str(TEST_IMAGE_DIR),
    imgsz=704,
    conf=0.10,
    iou=0.65,
    max_det=200,
    device=0,

    save=True,
    save_txt=True,
    save_conf=True,

    project=str(PREDICT_DIR),
    name="single_flower_yolo11s_v1_conf010",
    exist_ok=True,
)

RESULT_DIR = PREDICT_DIR / "single_flower_yolo11s_v1_conf010"
print("결과 저장:", RESULT_DIR)


image 1/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/#fleurs #bouquet.jpeg: 704x544 35 flowers, 62.2ms
image 2/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/_ (1).jpeg: 704x544 17 flowers, 15.2ms
image 3/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/_.jpeg: 704x576 30 flowers, 71.3ms
image 4/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/flowers.jpeg: 704x544 29 flowers, 15.9ms
image 5/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/test_1.jpeg: 544x704 56 flowers, 55.3ms
image 6/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/test_2.jpeg: 544x704 14 flowers, 15.2ms
image 7/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/test_3.jpeg: 704x544 51 flowers, 16.0ms
image 8/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/test_4.jpeg: 704x544 15 flowers, 15.2ms
image 9/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/머햇다구4월이_.

예측 이미지 확인

In [ ]:
image_paths = []

for ext in ["*.jpg", "*.jpeg", "*.png", "*.webp"]:
    image_paths.extend(RESULT_DIR.glob(ext))

print("결과 이미지 개수:", len(image_paths))

for img_path in image_paths[:30]:
    print(img_path.name)
    display(Image(filename=str(img_path), width=700))

In [4]:
from ultralytics import YOLO
from pathlib import Path

BEST_PT = Path("/content/drive/MyDrive/Single_Flower/training_results/single_flower_yolo11s_v1_704_baseline/weights/best.pt")
TEST_IMAGE_DIR = Path("/content/drive/MyDrive/Single_Flower/test_images_single_flower")
PREDICT_DIR = Path("/content/drive/MyDrive/Single_Flower/prediction_results")

model = YOLO(str(BEST_PT))

settings = [
    {"conf": 0.10, "iou": 0.45, "name": "conf010_iou045"},
    {"conf": 0.15, "iou": 0.45, "name": "conf015_iou045"},
    {"conf": 0.25, "iou": 0.40, "name": "conf025_iou040"},
    {"conf": 0.35, "iou": 0.35, "name": "conf035_iou035"},
]

for s in settings:
    results = model.predict(
        source=str(TEST_IMAGE_DIR),
        imgsz=704,
        conf=s["conf"],
        iou=s["iou"],
        max_det=120,
        device=0,

        save=True,
        save_txt=True,
        save_conf=True,

        project=str(PREDICT_DIR),
        name=f"single_flower_yolo11s_v1_{s['name']}",
        exist_ok=True,
    )

    print("saved:", s)


image 1/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/#fleurs #bouquet.jpeg: 704x544 30 flowers, 54.6ms
image 2/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/_ (1).jpeg: 704x544 16 flowers, 15.2ms
image 3/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/_.jpeg: 704x576 27 flowers, 53.7ms
image 4/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/flowers.jpeg: 704x544 26 flowers, 16.0ms
image 5/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/test_1.jpeg: 544x704 50 flowers, 66.0ms
image 6/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/test_2.jpeg: 544x704 12 flowers, 15.2ms
image 7/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/test_3.jpeg: 704x544 43 flowers, 15.9ms
image 8/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/test_4.jpeg: 704x544 15 flowers, 15.3ms
image 9/9 /content/drive/MyDrive/Single_Flower/test_images_single_flower/머햇다구4월이_.

In [6]:
from pathlib import Path
from PIL import Image as PILImage, ImageDraw, ImageFont
from IPython.display import display

PREDICT_DIR = Path("/content/drive/MyDrive/Single_Flower/prediction_results")

result_dirs = {
    "conf010_iou045": PREDICT_DIR / "single_flower_yolo11s_v1_conf010_iou045",
    "conf015_iou045": PREDICT_DIR / "single_flower_yolo11s_v1_conf015_iou045",
    "conf025_iou040": PREDICT_DIR / "single_flower_yolo11s_v1_conf025_iou040",
    "conf035_iou035": PREDICT_DIR / "single_flower_yolo11s_v1_conf035_iou035",
}

def list_image_files(folder):
    folder = Path(folder)
    paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.webp"]:
        paths.extend(folder.glob(ext))
    return sorted(paths)

# 첫 번째 폴더 기준으로 이미지 이름 목록 생성
base_key = "conf010_iou045"
base_images = list_image_files(result_dirs[base_key])
image_names = [p.name for p in base_images]

print("비교 가능한 이미지 개수:", len(image_names))
print(image_names[:10])

비교 가능한 이미지 개수: 9
['#fleurs #bouquet.jpg', '_ (1).jpg', '_.jpg', 'flowers.jpg', 'test_1.jpg', 'test_2.jpg', 'test_3.jpg', 'test_4.jpg', '머햇다구4월이_.jpg']


In [ ]:
def make_comparison_grid(image_name, width_each=360):
    panels = []

    for label, folder in result_dirs.items():
        img_path = folder / image_name

        if not img_path.exists():
            continue

        img = PILImage.open(img_path).convert("RGB")

        ratio = width_each / img.width
        new_h = int(img.height * ratio)
        img = img.resize((width_each, new_h))

        header_h = 38
        canvas = PILImage.new("RGB", (width_each, new_h + header_h), "white")
        canvas.paste(img, (0, header_h))

        draw = ImageDraw.Draw(canvas)
        draw.text((8, 10), label, fill=(0, 0, 0))

        panels.append(canvas)

    if not panels:
        return None

    total_w = sum(p.width for p in panels)
    max_h = max(p.height for p in panels)

    grid = PILImage.new("RGB", (total_w, max_h), "white")

    x = 0
    for p in panels:
        grid.paste(p, (x, 0))
        x += p.width

    return grid


# 앞에서부터 10장 비교
for image_name in image_names[:10]:
    print(image_name)
    grid = make_comparison_grid(image_name, width_each=320)
    if grid:
        display(grid)